## 基本関数定義、データ生成部分

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cupy as cp

N = 40
F = 8.0

def L96(x,t,F = 8):
  return (cp.roll(x,-1,axis = 0) - cp.roll(x,2, axis = 0)) * cp.roll(x,1,axis = 0) - x + F

def rk4(f, x0, t, F=8.0):
    # len(x0) ではなく *x0.shape を使うことで、1次元でも2次元でも柔軟にサイズを合わせる
    x = cp.zeros((len(t), *x0.shape))
    x[0] = x0
    for i in range(1, len(t)):
        dt = t[i] - t[i-1]
        k1 = f(x[i-1], t[i-1], F)
        k2 = f(x[i-1] + 0.5 * dt * k1, t[i-1] + 0.5 * dt, F)
        k3 = f(x[i-1] + 0.5 * dt * k2, t[i-1] + 0.5 * dt, F)
        k4 = f(x[i-1] + dt * k3, t[i-1] + dt, F)
        x[i] = x[i-1] + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
    return x

rng = cp.random.default_rng(seed=42)

## 真値と観測データの生成
x_minus90 = cp.full(N,F)
x_minus90[19] = F + 1.001
spinup_days = 360
t_spinup = cp.arange(0.0,spinup_days / 5,0.05)
x_spinup = rk4(L96, x_minus90, t_spinup, F=F)
t_truth = cp.arange(0.0,spinup_days / 5,0.05)
x_truth = rk4(L96, x_spinup[-1], t_truth, F=F)
y_observe = x_truth + rng.standard_normal(size=x_truth.shape)

H = cp.eye(N)
R = cp.eye(N) * 1.0
R_inv = cp.linalg.inv(R)
ensembles = 20
sigma_loc = 4
curr_X_analysis = cp.zeros((N,ensembles))

noise = rng.standard_normal(size=(N,ensembles))
noise = noise - cp.mean(noise, axis=1, keepdims=True)  #ノイズの平均が厳密に0になるようにしとく
for i in range(ensembles):
  member_x = x_spinup[-1] + noise[:, i]
  member_x_spinup = rk4(L96, member_x, t_spinup, F=F)
  curr_X_analysis[:, i] = member_x_spinup[-1]





ModuleNotFoundError: No module named 'cupy'

## パターン1 : GPU無し

In [ ]:
history_X_analysis = [curr_X_analysis.copy()] # アンサンブルメンバーの解析値を格納するリスト, 全時刻分
for step in range(1, len(t_truth)):
    t_step = t_truth[step-1:step+1]

    curr_X_background = rk4(L96, curr_X_analysis, t_step, F=F)[-1]

    x_background_mean = cp.mean(curr_X_background, axis=1)
    Z_background = (curr_X_background - x_background_mean[:, None]) / cp.sqrt(ensembles - 1)
    Y_background = H @ Z_background

    d_ob = y_observe[step] - H @ x_background_mean
    next_X_analysis = cp.zeros_like(curr_X_analysis)

   # (2) 局所解析ステップ: 各格子点(変数)ごとのループ (内側のループ変数を j に変更)
    for j in range(N):
        # 2.1 局所化 (Localization)
        # 周期境界条件を考慮して、着目格子点 j から各観測点までの最短距離を計算
        dist = cp.minimum(cp.abs(cp.arange(N) - j), N - cp.abs(cp.arange(N) - j))
        # ガウス関数による局所化重みの計算
        loc_weight = cp.exp(-(dist**2) / (2 * sigma_loc**2))
        
        # 局所化された観測誤差分散行列の逆行列を生成
        R_loc_inv = cp.diag(loc_weight) @ R_inv

        # 2.2 アンサンブル空間での解析誤差共分散 (P~a) の逆行列を計算
        # (P~a)^-1 = I + (Y^b)^T * R_loc^-1 * Y^b
        Pa_inv = cp.eye(ensembles) + Y_background.T @ R_loc_inv @ Y_background

        # 2.3 固有値分解 (EVD) による平方根行列の計算
        eigenvalues, eigenvectors = cp.linalg.eigh(Pa_inv)
        
        # P~a とその平方根行列 (P~a)^1/2 を構築
        Pa = eigenvectors @ cp.diag(1.0 / eigenvalues) @ eigenvectors.T
        Pa_sqrt = eigenvectors @ cp.diag(1.0 / cp.sqrt(eigenvalues)) @ eigenvectors.T

        # 2.4 変換行列 T の計算
        # 平均の更新用ウェイトベクトル (サイズ: m)
        # w~ = P~a * (Y^b)^T * R_loc^-1 * d_ob
        w_mean = Pa @ Y_background.T @ R_loc_inv @ d_ob

        # 摂動の更新用ウェイト行列 (サイズ: m x m)
        # W~ = sqrt(m-1) * (P~a)^1/2
        W_pert = cp.sqrt(ensembles - 1) * Pa_sqrt

        # 合成して T行列を作成: T = w~ * 1^T + W~
        T_matrix = cp.outer(w_mean, cp.ones(ensembles)) + W_pert

        # 2.5 アンサンブルの更新 (着目している格子点 j のみ更新)
        # X^a_j = x^b_mean_j + Z^b_j * T
        next_X_analysis[j, :] = x_background_mean[j] + Z_background[j, :] @ T_matrix

    # 解析結果を更新して保存
    curr_X_analysis = next_X_analysis
    history_X_analysis.append(curr_X_analysis.copy())
        

## パターン2 : GPUブースト

In [ ]:
history_X_analysis = [curr_X_analysis.copy()] # アンサンブルメンバーの解析値を格納するリスト, 全時刻分
# === ループ前の準備（事前計算） ===
# 局所化の重みは時間変化しないため、事前に 40x40 の行列としてすべて計算しておく
loc_weight_all = cp.zeros((N, N))
for j in range(N):
    dist = cp.minimum(cp.abs(cp.arange(N) - j), N - cp.abs(cp.arange(N) - j))
    loc_weight_all[j] = cp.exp(-(dist**2) / (2 * sigma_loc**2))

# 観測誤差 R が対角行列であることを前提に、対角成分だけを取り出す
R_inv_diag = cp.diag(R_inv)

# 最終的な重み行列 (サイズ: N x N)
# [j, i] 成分は、格子点 j における 観測点 i の重み (R_loc_invの対角成分) を表す
weight_matrix = loc_weight_all * R_inv_diag[None, :]

# =================================

for step in range(1, len(t_truth)):  # 注意: 初期値は計算済みなので1からスタート
    t_step = t_truth[step-1:step+1]
    curr_X_background = rk4(L96, curr_X_analysis, t_step, F=F)[-1]

    x_background_mean = cp.mean(curr_X_background, axis=1)
    Z_background = (curr_X_background - x_background_mean[:, None]) / cp.sqrt(ensembles - 1)
    Y_background = H @ Z_background

    d_ob = y_observe[step] - H @ x_background_mean

    # ----------------------------------------------------
    # (2) 局所解析ステップ: for j in range(N): を排除した完全バッチ処理
    # ----------------------------------------------------
    
    # 2.1 & 2.2 アンサンブル空間での解析誤差共分散 (P~a) の逆行列を計算
    # 40個の格子点すべての (m x m) 行列を一度に計算 (cp.einsum を活用)
    # 結果のサイズ: (N, m, m)
    Pa_inv = cp.eye(ensembles)[None, :, :] + cp.einsum('ji, im, in -> jmn', weight_matrix, Y_background, Y_background)

    # 2.3 固有値分解 (EVD)
    # CuPy の eigh はバッチ処理に対応しているため、(N, m, m) を渡せば40個同時に分解してくれる
    eigenvalues, eigenvectors = cp.linalg.eigh(Pa_inv)

    # P~a と (P~a)^1/2 をバッチ構築 (サイズ: N, m, m)
    D_inv = 1.0 / eigenvalues
    D_inv_sqrt = 1.0 / cp.sqrt(eigenvalues)
    Pa = eigenvectors @ (D_inv[..., None] * cp.swapaxes(eigenvectors, 1, 2))
    Pa_sqrt = eigenvectors @ (D_inv_sqrt[..., None] * cp.swapaxes(eigenvectors, 1, 2))

    # 2.4 変換行列 T の計算
    # 各格子点の重み付けされたイノベーションを計算 (サイズ: N, N)
    weighted_dob = weight_matrix * d_ob[None, :]
    
    # 平均の更新用ウェイトベクトル w~ (サイズ: N, m)
    Y_weighted_dob = cp.einsum('im, ji -> jm', Y_background, weighted_dob)
    w_mean = cp.einsum('jmn, jn -> jm', Pa, Y_weighted_dob)

    # 摂動の更新用ウェイト行列 W~ (サイズ: N, m, m)
    W_pert = cp.sqrt(ensembles - 1) * Pa_sqrt

    # 合成して T行列を作成: (サイズ: N, m, m)
    # NumPy/CuPyのブロードキャスト機能により、w_mean[:, :, None] は自動的に各列に足される
    T_matrix = w_mean[:, :, None] + W_pert

    # 2.5 アンサンブルの更新
    # すべての変数 j のアンサンブルを一度に更新 (サイズ: N, m)
    next_X_analysis = x_background_mean[:, None] + cp.einsum('jm, jmn -> jn', Z_background, T_matrix)

    # 解析結果を更新して保存
    curr_X_analysis = next_X_analysis
    history_X_analysis.append(curr_X_analysis.copy())


final_analysis_trajectory = cp.mean(cp.array(history_X_analysis), axis=2) # サイズ: [Time, N]

In [ ]:
#print(final_analysis_trajectory.shape)
print("RMSE :" + str(cp.sqrt(cp.mean((final_analysis_trajectory - x_truth)**2))))

RMSE :0.23119929687492854
